In [ ]:
#!pip install python-docx openpyxl
#!pip install python-docx docx2pdf pandas openpyxl
#!pip install docx2pdf

#!pip install --upgrade xlrd


Defaulting to user installation because normal site-packages is not writeable
  Attempting uninstall: xlrd
    Found existing installation: xlrd 0.7.1
    Uninstalling xlrd-0.7.1:
      Successfully uninstalled xlrd-0.7.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
camelot 12.6.29 requires xlrd==0.7.1, but you have xlrd 2.0.2 which is incompatible.


# Guía didáctica

In [18]:
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.oxml import parse_xml
from docx.oxml.ns import nsdecls
from docx.enum.text import WD_ALIGN_PARAGRAPH  # para la justificación
import pandas as pd
from datetime import datetime
import os

# IMPORTANTE: La librería docx2pdf requiere tener Microsoft Word instalado y configurado correctamente.
from docx2pdf import convert

# -------------------------
# Rutas de archivos
# -------------------------
plantilla_path = "G:/Mi unidad/FACEN_BIGDATA/BigData_plantilla_Guía_Didáctica.docx"
excel_path = "G:/Mi unidad/FACEN_BIGDATA/FACEN_BigData_GD_VF.xlsx"
# Se eliminan acentos para evitar problemas durante la conversión
output_word_path = "G:/Mi unidad/FACEN_BIGDATA/BigData_GuiaDid.docx"
output_pdf_path = "G:/Mi unidad/FACEN_BIGDATA/BigData_GuiaDid.pdf"

# -------------------------
# Función para ajustar automáticamente el ancho de las columnas
# -------------------------
def auto_adjust_column_widths(table, df, factor=0.12):
    """
    Ajusta el ancho de las columnas de la tabla en función del contenido
    y el encabezado, usando un factor de escala para convertir "caracteres" a "pulgadas".
    """
    for i, col in enumerate(df.columns):
        max_length = max([len(str(val)) for val in df[col]] + [len(str(col))])
        width = Inches(max_length * factor)
        for cell in table.columns[i].cells:
            cell.width = width

# -------------------------
# 1. Crear documento a partir de la plantilla
# -------------------------
doc = Document(plantilla_path)

# -------------------------
# 2. Crear portada
# -------------------------
cover_texts = [
    "",
    "",
    "",
    "Analítica de Big Data", 
    "Guía Didáctica",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "Campus Universitario",
    "San Lorenzo, Paraguay",
    "2025"]

for text in cover_texts:
    p = doc.add_paragraph()
    run = p.add_run(text)
    if text in ["Analítica de Big Data", "Guía Didáctica"]:
        run.font.size = Pt(38)
        run.font.color.rgb = RGBColor(102, 51, 0)  # Naranja oscuro
    elif text in ["Campus Universitario", "San Lorenzo, Paraguay", "2025"]:
        run.font.size = Pt(18)
        run.font.color.rgb = RGBColor(0, 0, 0)  # Negro
    else:
        run.font.size = Pt(12)  # Tamaño por defecto
        run.font.color.rgb = RGBColor(0, 0, 0)
    p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    p.paragraph_format.first_line_indent = Inches(0.5)


# Salto de página
doc.add_page_break()

# -------------------------
# 3. Insertar contenido principal
# -------------------------
def add_paragraph_with_format(text, style=None):
    """
    Agrega un párrafo con justificación y sangría de primera línea.
    """
    p = doc.add_paragraph(text, style=style)
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    p.paragraph_format.first_line_indent = Inches(0.5)
    return p

# 1. Presentación del curso
doc.add_heading("1. Presentación del curso", level=1)
add_paragraph_with_format(
    "El Big Data es una forma de aplicar la ciencia de datos, específicamente cuando se requiere explorar grandes cantidades de datos generados continuamente y con gran velocidad al interior de las empresas o instituciones. Estos datos presentan características particulares que impiden su análisis con técnicas tradicionales, lo que impulsa la evolución de nuevas metodologías, entre ellas las conocidas como Analítica de Big Data. Este curso se desarrolla para proporcionar a los participantes las herramientas necesarias para el manejo y análisis de dichos datos."
)

# 2. Objetivos
doc.add_heading("2. Objetivos", level=1)
doc.add_heading("2.1 Objetivo General", level=2)
add_paragraph_with_format(
    "Otorgar a los participantes conocimientos para la gestión y el análisis estadístico de grandes volúmenes de datos o Big Data, mediante herramientas open source como R."
)
doc.add_heading("2.2 Objetivos Específicos", level=2)
objetivos = [
    "Identificar algunos de los programas informáticos que permiten encarar el análisis estadístico del Big Data.",
    "Identificar los pasos y las herramientas para la recopilación y gestión de datos con las características particulares del Big Data.",
    "Emplear las técnicas adecuadas para visualizar y explorar el Big Data.",
    "Identificar y utilizar algunas herramientas para la técnica de aprendizaje automático (Machine Learning).",
    "Describir e interpretar los resultados obtenidos mediante técnicas de modelado estadístico en Big Data."
]
for obj in objetivos:
    add_paragraph_with_format(obj)

# 3. Metodología
doc.add_heading("3. Metodología", level=1)
add_paragraph_with_format(
    "La asignatura se desarrollará en modalidad semipresencial, combinando actividades presenciales (encuentros con el tutor y exámenes finales) y no presenciales a través de la plataforma virtual Moodle. Se utilizarán herramientas como foros, cuestionarios, tareas, etc. El ingreso a la plataforma se realizará con usuario y contraseña asignados a cada estudiante. Las unidades se habilitarán gradualmente y cada una contará con orientaciones detalladas sobre las actividades y plazos."
)

# 4. Contenido
doc.add_heading("4. Contenido", level=1)
contenido = {
    "Unidad 1: Manejo de datos en Big Data con R": [
         "Instalación e interface",
         "Carga y uso de paquetes",
         "Operaciones básicas y gráficos",
         "Gestión de archivos y bases de datos",
         "Análisis exploratorios de datos",
         "Los paquetes de R para Big Data"
    ],
    "Unidad 2: Visualización y exploración de datos con R": [
         "Binarización de variables cualitativas",
         "Selección de predictores",
         "Tipo de variables y distribución",
         "Distribución de variables de respuesta discreta",
         "Exploración de datos",
         "División de los datos en entrenamiento y test",
         "Preprocesado de los datos",
         "Imputación de valores ausentes",
         "Variables con varianza a cero",
         "Estandarización y escalado",
         "Árbol de clasificación simple",
         "Random Forest"
    ],
    "Unidad 3: Machine Learning con R": [
         "Creación de modelo predictivo",
         "K-Nearest Neighbor (KNN)",
         "Naive Bayes",
         "Regresión Logística"
    ],
    "Unidad 4: Interpretación de los resultados y modelos estadísticos obtenidos con R": [
         "Análisis Discriminante Lineal (LDA)",
         "Support Vector Machine",
         "Redes Neuronales (NNET)",
         "Gradient Boosting",
         "Interpretación y selección de resultados y los modelos para generar información estadísticas en Big Data"
    ]
}
unidad_num = 1
for unidad, temas in contenido.items():
    doc.add_heading(f"4.{unidad_num} {unidad}", level=2)
    tema_num = 1
    for tema in temas:
        add_paragraph_with_format(f"4.{unidad_num}.{tema_num} {tema}")
        tema_num += 1
    unidad_num += 1

# Salto de página
doc.add_page_break()
# 5. Cronograma
doc.add_heading("5. Cronograma", level=1)

# Leer datos desde Excel


# -------------------------
# 5. Cronograma
# -------------------------


# Leer datos desde Excel (forzamos engine openpyxl)
cronograma_df = pd.read_excel(
    excel_path,
    sheet_name='cronograma',
    usecols='A:K',
    skiprows=17,
    nrows=26,
    engine='openpyxl'
)

# Eliminar columnas no deseadas
cronograma_df = cronograma_df.drop(
    columns=['Objetivo de la unidad', 'Duración (días)', 'Objetivo de la actividad']
)

# Renombrar columnas para que coincidan con el formato del PDF
column_mapping = {
    "Unidad": "Unidad",
    "Nombre unidad": "Actividad",
    "Descripcion de la actividad": "Nombre act.",
    "Tipo de actividad": "Tipo act.",
    "Puntaje asignado": "Puntaje",
    "Fecha inicio": "Fecha inicio",
    "Fecha fin": "Fecha fin"
}
cronograma_df.rename(columns=column_mapping, inplace=True)

# Crear tabla en el documento (una fila para el encabezado)
table = doc.add_table(rows=1, cols=len(cronograma_df.columns))
table.autofit = True

# Encabezado: fondo oscuro y letras blancas
hdr_cells = table.rows[0].cells
for i, col_name in enumerate(cronograma_df.columns):
    hdr_cells[i].text = col_name
    run = hdr_cells[i].paragraphs[0].runs[0]
    run.font.bold = True
    run.font.size = Pt(9)
    run.font.color.rgb = RGBColor(255, 255, 255)
    shading_elm = parse_xml(r'<w:shd {} w:fill="654321"/>'.format(nsdecls('w')))
    hdr_cells[i]._element.get_or_add_tcPr().append(shading_elm)

# Contenido de la tabla (filas del DataFrame)
for idx, row in cronograma_df.iterrows():
    cells = table.add_row().cells
    for i, value in enumerate(row):
        if isinstance(value, datetime):
            text_value = value.strftime("%d/%m/%Y")
        else:
            text_value = str(value)
        cells[i].text = text_value
        for paragraph in cells[i].paragraphs:
            for run in paragraph.runs:
                run.font.size = Pt(9)
    fill_color = "FFEFD5" if idx % 2 == 0 else "FFFFFF"
    for cell in cells:
        cell_shading = parse_xml(r'<w:shd {} w:fill="{}"/>'.format(nsdecls('w'), fill_color))
        cell._element.get_or_add_tcPr().append(cell_shading)

# Bordes de la tabla en color gris oscuro
tbl = table._tbl
tblPr = tbl.tblPr
tblBorders = parse_xml(
    r'<w:tblBorders {}>'
    r'<w:top w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
    r'<w:left w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
    r'<w:bottom w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
    r'<w:right w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
    r'<w:insideH w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
    r'<w:insideV w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
    r'</w:tblBorders>'.format(nsdecls('w'))
)
tblPr.append(tblBorders)
auto_adjust_column_widths(table, cronograma_df)

# 6. Carga horaria
doc.add_heading("6. Carga horaria", level=1)
add_paragraph_with_format("Total de horas del curso: 60 horas reloj.")

# 7. Actividades para el estudiante
doc.add_heading("7. Actividades para el estudiante", level=1)
add_paragraph_with_format(
    "En cada unidad de la asignatura el estudiante deberá realizar una o más actividades. Entre ellas se encuentran las Actividades de Aprendizaje (AA), como las tareas para subir archivos (ensayos y resúmenes) o cuestionarios, y las Actividades Interactivas (AI), como una wiki o foros. Por otra parte, las Actividades de Fijación (AF) no son sumativas ni obligatorias, pero están vinculadas a las actividades de evaluación y son de gran importancia para la construcción de los aprendizajes."
)

# 8. Asesoría de apoyo al aprendizaje de los estudiantes
doc.add_heading("8. Asesoría de apoyo al aprendizaje de los estudiantes", level=1)
add_paragraph_with_format(
    "Las asesorías y tutorías se realizarán a través de foros de consultas, que serán habilitados para cada unidad a desarrollarse. Adicionalmente, se tendrá una tutoría sincrónica por cada unidad a desarrollarse en las fechas indicadas, los días sábado de 18 a 20 horas."
)

# 9. Evaluación
doc.add_heading("9. Evaluación", level=1)
add_paragraph_with_format(
    "La evaluación de la asignatura es continua. Todos los trabajos presentados tendrán un periodo de evaluación por parte de la tutora, quien al finalizarlo realizará una devolución escrita con las revisiones y sugerencias. Para las evaluaciones del proceso y finales se tendrán en cuenta lo establecido en el Reglamento Académico (Artículo 15(B)): "
    "las evaluaciones de proceso tienen una ponderación de 40 % y las evaluaciones finales del 60 %."
)

# Insertar tabla de evaluación (desglose de ponderación)
table_eval = doc.add_table(rows=6, cols=2)
table_eval.autofit = True

# Encabezados de la tabla
hdr_cells = table_eval.rows[0].cells
hdr_cells[0].text = "Concepto"
hdr_cells[1].text = "Ponderación"

data_eval = [
    ["Actividad de aprendizaje (envío de tareas)", "15 %"],
    ["Participación en actividades colaborativas", "15 %"],
    ["Evaluación Parcial", "10 %"],
    ["Evaluación Final", "60 %"],
    ["Total", "100 %"]
]

for i, row_data in enumerate(data_eval, start=1):
    row_cells = table_eval.rows[i].cells
    row_cells[0].text = row_data[0]
    row_cells[1].text = row_data[1]

# Separador antes de la siguiente tabla
doc.add_paragraph("Las calificaciones finales se regirán por la siguiente escala:")

# Insertar tabla de escala de calificaciones
table_scale = doc.add_table(rows=1, cols=4)
table_scale.autofit = True

headers_scale = ["Aprobado/Reprobado", "Rango de puntos", "Calificación numérica", "Calificación cualitativa"]
hdr_cells_scale = table_scale.rows[0].cells
for j, header in enumerate(headers_scale):
    hdr_cells_scale[j].text = header

data_scale = [
    ["Reprobado", "1 % - 59 %", "1 (Uno)", "Insuficiente"],
    ["Aprobado", "60 % - 70 %", "2 (dos)", "Aceptable"],
    ["Aprobado", "71 % - 80 %", "3 (tres)", "Bueno"],
    ["Aprobado", "81 % - 90 %", "4 (cuatro)", "Distinguido"],
    ["Aprobado", "91 % - 100 %", "5 (cinco)", "Sobresaliente"]
]

for row_data in data_scale:
    row_cells = table_scale.add_row().cells
    for j, cell_text in enumerate(row_data):
        row_cells[j].text = cell_text

# 10. Recomendaciones y consideraciones finales
doc.add_heading("10. Recomendaciones y consideraciones finales", level=1)
add_paragraph_with_format(
    "La modalidad a distancia, si bien ofrece flexibilidad, requiere un gran esfuerzo y organización por parte del estudiante. Se recomienda una dedicación mínima semanal de 5 horas. Es fundamental prestar atención a las condiciones de entrega de tareas, realizar la lectura de los materiales de forma reflexiva y valorar cada experiencia de aprendizaje. El desarrollo de las Actividades de Fijación (AF) es crucial, ya que favorece el logro de las actividades sumativas."
)

# 11. Referencias
doc.add_heading("Referencias", level=1)
referencias = [
    "[1] Microsoft. Ayuda y formación de Excel. Microsoft Excel, 2022. https://support.microsoft.com/es-es/excel",
    "[2] Ahumada, J. A. (2003). R. para principiantes. University of Hawaii, 2003.",
    "[3] Van Rossum, G., Drake, F. L. (2009). Python 3 Reference Manual. Scotts Valley, CA: CreateSpace, 2009."
]
for ref in referencias:
    add_paragraph_with_format(ref)

# -------------------------
# 12. Guardar el documento Word
# -------------------------
doc.save(output_word_path)
print(f"Documento Word generado exitosamente en: {output_word_path}")

# -------------------------
# 13. Convertir a PDF usando docx2pdf
# -------------------------
try:
    convert(output_word_path, output_pdf_path)
    print(f"Documento PDF generado exitosamente en: {output_pdf_path}")
except Exception as e:
    print("Ocurrió un error al convertir a PDF:", str(e))


Documento Word generado exitosamente en: G:/Mi unidad/FACEN_BIGDATA/BigData_GuiaDid.docx


  0%|          | 0/1 [00:00<?, ?it/s]

Documento PDF generado exitosamente en: G:/Mi unidad/FACEN_BIGDATA/BigData_GuiaDid.pdf


In [19]:
cronograma_df

,Unidad,Actividad,Contenidos,Nombre act.,Tipo act.,Puntaje,Fecha inicio,Fecha fin
0,1,Introducción al manejo de datos en Big Data,Orígen e Introducciones conceptuales a la Anal...,1. Estudio de los materiales de aprendizaje,AF,0,2025-08-04,2025-08-06
1,1,Introducción al manejo de datos en Big Data,Instalación y configuración de R y Rstudio,2. Foro de discusión sobre problemas de aplica...,AI,100,2025-08-07,2025-08-12
2,1,Introducción al manejo de datos en Big Data,Operacione básicas y gráficos,3. Desarrollo del cuestionario en línea,AA,100,2025-08-13,2025-08-18
3,2,Visualización y exploración de datos,"Librerías de R para big data, carga y procesam...",1. Estudio de los materiales de aprendizaje,AF,0,2025-08-19,2025-08-21
4,2,Visualización y exploración de datos,Binarización de variables cualitativas. Divisi...,2. Foro de discusión sobre problemas de aplica...,AI,100,2025-08-21,2025-08-30
5,2,Visualización y exploración de datos,"Técnicas de imputación, estandarización y esca...",3. Desarrollo del cuestionario en línea,AA,100,2025-08-31,2025-09-09
6,3,Machine Learning,Creación de modelo predictivo,1. Estudio de los materiales de aprendizaje,AF,0,2025-09-29,2025-10-03
7,3,Machine Learning,K-Nearest Neighbor (KNN),2. Foro de discusión sobre problemas de aplica...,AI,100,2025-10-03,2025-10-06
8,3,Machine Learning,Regresión Logística,3. Desarrollo del cuestionario en línea,AA,100,2025-10-07,2025-10-09
9,4,Interpretación de los resultados y modelos est...,Interpretación y selección de resultados y los...,1. Estudio de los materiales de aprendizaje,AI,0,2025-10-10,2025-10-14


# Orientaciones de las unidades

In [21]:
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.oxml import parse_xml
from docx.oxml.ns import nsdecls
from docx.enum.text import WD_ALIGN_PARAGRAPH
import pandas as pd
from datetime import datetime

# Importar docx2pdf para la conversión
from docx2pdf import convert

# Función auxiliar para agregar párrafos con justificación y sangría a un documento
def add_paragraph_with_format_to(doc_obj, text, style=None):
    p = doc_obj.add_paragraph(text, style=style)
    p.alignment = WD_ALIGN_PARAGRAPH.LEFT
    p.paragraph_format.first_line_indent = Inches(0.5)
    return p

# Función para ajustar automáticamente el ancho de las columnas
def auto_adjust_column_widths(table, df, factor=0.12):
    for i, col in enumerate(df.columns):
        max_length = max([len(str(val)) for val in df[col]] + [len(str(col))])
        width = Inches(max_length * factor)
        for cell in table.columns[i].cells:
            cell.width = width

# =========================
# Rutas de archivos para orientaciones
# =========================
template_orientaciones_path = "G:/Mi unidad/FACEN_BIGDATA/BigData_plantilla_OrientacionesUnidad.docx"
output_orientation_base = "G:/Mi unidad/FACEN_BIGDATA/Orientaciones_Unidad_"
excel_path = "G:/Mi unidad/FACEN_BIGDATA/FACEN_BigData_GD_VF.xlsx"

# =========================
# Información de orientaciones para cada unidad
# =========================
orientaciones = {
    1: {
        "nombre": "Introducción al manejo de datos en Big Data",
        "objetivo": "Conocer los conceptos básicos y herramientas para el manejo de datos en Big Data.",
        "cron": "Ejercicios prácticos usando Excel y R"
    },
    2: {
        "nombre": "Visualización y exploración de datos",
        "objetivo": "Comprender técnicas de visualización y análisis exploratorio.",
        "cron": "Ejercicios prácticos en la elaboración de gráficos y análisis de distribuciones."
    },
    3: {
        "nombre": "Machine Learning",
        "objetivo": "Implementar modelos predictivos utilizando algoritmos de Machine Learning.",
        "cron": "Actividades de modelado, pruebas y validación de algoritmos."
    },
    4: {
        "nombre": "Interpretación de los resultados y modelos estadísticos",
        "objetivo": "Analizar y comunicar resultados de modelos estadísticos aplicados a Big Data.",
        "cron": "Ejercicios de interpretación y presentación de resultados."
    },
    5: {
        "nombre": "Evaluaciones parciales y Finales",
        "objetivo": "Integrar y aplicar los conocimientos adquiridos a través de examenes y la entrega de un trabajo final.",
        "cron": "Desarrollo y entrega de un trabajo práctico final que sintetice lo aprendido."
    }
}

# =========================
# Generar documentos independientes de Orientaciones para cada unidad
# =========================
for unidad_codigo, unit in orientaciones.items():
    # Abrir la plantilla de orientaciones
    doc_orient = Document(template_orientaciones_path)
    
    # Encabezado / Portada del documento
    h = doc_orient.add_heading("Departamento de Educación a Distancia", level=1)
    h.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    h = doc_orient.add_heading("Analítica de Big Data", level=1)
    h.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Línea en blanco (opcional)
    h = doc_orient.add_heading("", level=1)
    h.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    h = doc_orient.add_heading(f"Unidad {unidad_codigo}: {unit['nombre']}", level=2)
    h.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    h = doc_orient.add_heading("Orientaciones", level=2)
    h.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Contenido principal
    add_paragraph_with_format_to(doc_orient, "En esta unidad desarrollarás contenidos para lograr el siguiente objetivo:")
    add_paragraph_with_format_to(doc_orient, f"- {unit['objetivo']}")
    add_paragraph_with_format_to(doc_orient, "Las actividades que se realizarán para el logro de este objetivo son:")
    
    # Leer cronograma desde Excel
    cronograma_df = pd.read_excel(
        excel_path,
        sheet_name='cronograma',
        usecols='A:K',
        skiprows=17,
        nrows=26
    )
    # Eliminar columnas no deseadas
    cronograma_df = cronograma_df.drop(
        columns=['Objetivo de la unidad', 'Duración (días)', 'Objetivo de la actividad']
    )
    
    # Filtrar por la columna "Nombre unidad"
    if "Nombre unidad" in cronograma_df.columns:
        # Eliminamos espacios y buscamos si el nombre de la unidad coincide
        filtro = cronograma_df["Nombre unidad"].astype(str).str.strip().str.contains(
            unit["nombre"].strip(), case=False
        )
        cronograma_unit = cronograma_df.loc[filtro]
    else:
        print(f"[Advertencia] La columna 'Nombre unidad' no se encontró en el cronograma. Se incluirán todas las filas para la Unidad {unidad_codigo}.")
        cronograma_unit = cronograma_df

    # Crear la tabla en el documento (solo si hay datos para esa unidad)
    if not cronograma_unit.empty:
        table = doc_orient.add_table(rows=1, cols=len(cronograma_unit.columns))
        table.autofit = True

        # Encabezado con fondo oscuro y letras blancas
        hdr_cells = table.rows[0].cells
        for j, col_name in enumerate(cronograma_unit.columns):
            hdr_cells[j].text = col_name
            run = hdr_cells[j].paragraphs[0].runs[0]
            run.font.bold = True
            run.font.size = Pt(9)
            run.font.color.rgb = RGBColor(255, 255, 255)
            shading_elm = parse_xml(r'<w:shd {} w:fill="654321"/>'.format(nsdecls('w')))
            hdr_cells[j]._element.get_or_add_tcPr().append(shading_elm)
        
        # Insertar datos (filas) en la tabla
        for idx, row in cronograma_unit.iterrows():
            cells = table.add_row().cells
            for j, value in enumerate(row):
                if isinstance(value, datetime):
                    text_value = value.strftime("%d/%m/%Y")
                else:
                    text_value = str(value)
                cells[j].text = text_value
                for paragraph in cells[j].paragraphs:
                    for run in paragraph.runs:
                        run.font.size = Pt(9)

            # Sombreado alternado
            fill_color = "FFEFD5" if idx % 2 == 0 else "FFFFFF"
            for cell in cells:
                cell_shading = parse_xml(r'<w:shd {} w:fill="{}"/>'.format(nsdecls('w'), fill_color))
                cell._element.get_or_add_tcPr().append(cell_shading)
        
        # Bordes de la tabla en color gris oscuro
        tbl = table._tbl
        tblPr = tbl.tblPr
        tblBorders = parse_xml(
            r'<w:tblBorders {}>'
            r'<w:top w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
            r'<w:left w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
            r'<w:bottom w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
            r'<w:right w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
            r'<w:insideH w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
            r'<w:insideV w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
            r'</w:tblBorders>'.format(nsdecls('w'))
        )
        tblPr.append(tblBorders)
        
        # Ajustar ancho de columnas
        auto_adjust_column_widths(table, cronograma_unit)
    else:
        add_paragraph_with_format_to(doc_orient, "No se encontraron datos en el cronograma para esta unidad.")
    
    # Mensajes finales
    add_paragraph_with_format_to(
        doc_orient,
        "Referencias: AA: Actividad de aprendizaje individual, AI: Actividad Interactiva, AF: Actividad de Fijación"
    )
    add_paragraph_with_format_to(
        doc_orient,
        "Estas fechas son recomendaciones para ayudarte a organizar tus estudios. Ante cualquier duda, recuerda que puedes manifestarla en el foro Consultas de esta unidad. Estoy aquí para acompañarte."
    )
    
    # Guardar el documento Word
    output_unit_docx = f"{output_orientation_base}Unidad{unidad_codigo}.docx"
    doc_orient.save(output_unit_docx)
    print(f"Documento de orientaciones (Word) para la Unidad {unidad_codigo} generado en: {output_unit_docx}")
    
    # =========================
    # Convertir a PDF con docx2pdf
    # =========================
    output_unit_pdf = f"{output_orientation_base}Unidad{unidad_codigo}.pdf"
    try:
        convert(input_path=output_unit_docx, output_path=output_unit_pdf)
        print(f"Documento de orientaciones (PDF) para la Unidad {unidad_codigo} generado en: {output_unit_pdf}\n")
    except Exception as e:
        print(f"[Error] No se pudo convertir a PDF para la Unidad {unidad_codigo}: {str(e)}\n")


Documento de orientaciones (Word) para la Unidad 1 generado en: G:/Mi unidad/FACEN_BIGDATA/Orientaciones_Unidad_Unidad1.docx


  0%|          | 0/1 [00:00<?, ?it/s]

Documento de orientaciones (PDF) para la Unidad 1 generado en: G:/Mi unidad/FACEN_BIGDATA/Orientaciones_Unidad_Unidad1.pdf

Documento de orientaciones (Word) para la Unidad 2 generado en: G:/Mi unidad/FACEN_BIGDATA/Orientaciones_Unidad_Unidad2.docx


  0%|          | 0/1 [00:00<?, ?it/s]

Documento de orientaciones (PDF) para la Unidad 2 generado en: G:/Mi unidad/FACEN_BIGDATA/Orientaciones_Unidad_Unidad2.pdf

Documento de orientaciones (Word) para la Unidad 3 generado en: G:/Mi unidad/FACEN_BIGDATA/Orientaciones_Unidad_Unidad3.docx


...

Documento de orientaciones (PDF) para la Unidad 3 generado en: G:/Mi unidad/FACEN_BIGDATA/Orientaciones_Unidad_Unidad3.pdf

Documento de orientaciones (Word) para la Unidad 4 generado en: G:/Mi unidad/FACEN_BIGDATA/Orientaciones_Unidad_Unidad4.docx


  0%|          | 0/1 [00:00<?, ?it/s]

Documento de orientaciones (PDF) para la Unidad 4 generado en: G:/Mi unidad/FACEN_BIGDATA/Orientaciones_Unidad_Unidad4.pdf

Documento de orientaciones (Word) para la Unidad 5 generado en: G:/Mi unidad/FACEN_BIGDATA/Orientaciones_Unidad_Unidad5.docx


  0%|          | 0/1 [00:00<?, ?it/s]

Documento de orientaciones (PDF) para la Unidad 5 generado en: G:/Mi unidad/FACEN_BIGDATA/Orientaciones_Unidad_Unidad5.pdf



# Descripciones de la actividades

In [23]:
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.oxml import parse_xml
from docx.oxml.ns import nsdecls
from docx.enum.text import WD_ALIGN_PARAGRAPH
import pandas as pd
from datetime import datetime
from docx2pdf import convert

def add_paragraph_with_format_to(doc_obj, text, style=None):
    p = doc_obj.add_paragraph(text, style=style)
    p.alignment = WD_ALIGN_PARAGRAPH.LEFT
    p.paragraph_format.first_line_indent = Inches(0.5)
    return p

def auto_adjust_column_widths(table, df, factor=0.12):
    for i, col in enumerate(df.columns):
        max_length = max([len(str(val)) for val in df[col]] + [len(str(col))])
        width = Inches(max_length * factor)
        for cell in table.columns[i].cells:
            cell.width = width

# Rutas de archivos
template_orientaciones_path = "G:/Mi unidad/FACEN_BIGDATA/BigData_plantilla_OrientacionesUnidad.docx"
excel_path = "G:/Mi unidad/FACEN_BIGDATA/FACEN_BigData_GD_VF.xlsx"
output_desc_base = "G:/Mi unidad/FACEN_BIGDATA/Descripcion_Actividad"

# Información de cada unidad
unidad_info = {
    1: {
        "nombre": "Introducción al manejo de datos en Big Data",
        "objetivo": "Conocer los conceptos básicos y herramientas para el manejo de datos en Big Data."
    },
    2: {
        "nombre": "Visualización y exploración de datos",
        "objetivo": "Comprender técnicas de visualización y análisis exploratorio."
    },
    3: {
        "nombre": "Machine Learning",
        "objetivo": "Implementar modelos predictivos utilizando algoritmos de Machine Learning."
    },
    4: {
        "nombre": "Interpretación de los resultados y modelos estadísticos",
        "objetivo": "Analizar y comunicar resultados de modelos estadísticos aplicados a Big Data."
    },
    5: {
        "nombre": "Entrega de trabajo practico final",
        "objetivo": "Integrar y aplicar los conocimientos adquiridos a través de un trabajo final."
    }
}

# Lista de actividades (3 actividades posibles)
activities = [
    (
        "Actividad 1: Estudio de los materiales de aprendizaje",
        (
            "Estimado estudiante, la actividad 1 consiste en estudiar los materiales disponibles en el apartado de Recursos del aula virtual.\n\n"
            "Para este trabajo, sigue las indicaciones:\n"
            "1. Estudia los contenidos tratados en clase apoyándote en los materiales de aprendizaje.\n"
            "2. Visualiza también los videos tutoriales disponibles.\n"
            "3. Resuelve las cuestiones planteadas en el apartado correspondiente dentro del aula virtual.\n"
            "4. El plazo para participar está indicado en la tabla."
        )
    ),
    (
        "Actividad 2: Foro de discusión sobre problemas de aplicación",
        (
            "Estimado estudiante, la actividad 2 consiste en participar en el foro de discusión sobre problemas de aplicación.\n\n"
            "Para este trabajo, sigue las indicaciones:\n"
            "1. Revisa los contenidos y reflexiona sobre los problemas planteados.\n"
            "2. Participa activamente en el foro comentando y respondiendo a las publicaciones.\n"
            "3. Argumenta tus respuestas con base en los contenidos vistos en clase.\n"
            "4. Consulta el cronograma para conocer el plazo de participación."
        )
    ),
    (
        "Actividad 3: Desarrollo del cuestionario en línea",
        (
            "Estimado estudiante, la actividad 3 consiste en desarrollar y completar el cuestionario en línea sobre lo aprendido.\n\n"
            "Para este trabajo, sigue las indicaciones:\n"
            "1. Revisa los materiales de aprendizaje y los ejemplos discutidos en clase.\n"
            "2. Accede al cuestionario en línea a través del aula virtual.\n"
            "3. Responde todas las preguntas con precisión.\n"
            "4. El plazo para completar esta actividad se encuentra en el cronograma."
        )
    )
]

# Generar documentos para cada actividad de cada unidad
for unit_num, info in unidad_info.items():
    # Leer y filtrar el cronograma desde Excel por unidad
    cronograma_df = pd.read_excel(
        excel_path,
        sheet_name='cronograma',
        usecols='A:K',
        skiprows=17,
        nrows=26
    )
    cronograma_df = cronograma_df.drop(
        columns=['Objetivo de la unidad', 'Duración (días)', 'Objetivo de la actividad']
    )
    if "Nombre unidad" in cronograma_df.columns:
        filtro_unidad = cronograma_df["Nombre unidad"].astype(str).str.strip().str.contains(
            info["nombre"].strip(), case=False
        )
        cronograma_unit = cronograma_df.loc[filtro_unidad]
    else:
        print(f"Advertencia: La columna 'Nombre unidad' no se encontró. Se usarán todos los datos para la Unidad {unit_num}.")
        cronograma_unit = cronograma_df

    for act_num, (act_title, act_instructions) in enumerate(activities, start=1):
        doc_desc = Document(template_orientaciones_path)
        # Encabezados principales
        h = doc_desc.add_heading("Departamento de Educación a Distancia", level=1)
        h.alignment = WD_ALIGN_PARAGRAPH.CENTER
        h = doc_desc.add_heading("Analítica de Big Data", level=1)
        h.alignment = WD_ALIGN_PARAGRAPH.CENTER
        doc_desc.add_paragraph("")  # Línea en blanco

        # Encabezado de la unidad
        h = doc_desc.add_heading(f"Unidad {unit_num}: {info['nombre']}", level=2)
        h.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Encabezado de la actividad con indexación corregida
        # Se extrae el título de la actividad (después de los dos puntos)
        activity_title = act_title.split(":", 1)[1].strip() if ":" in act_title else act_title.strip()
        h = doc_desc.add_heading(f"Actividad {unit_num}.{act_num}: {activity_title}", level=3)
        h.alignment = WD_ALIGN_PARAGRAPH.CENTER

        doc_desc.add_paragraph("")  # Línea en blanco

        # Instrucciones de la actividad
        add_paragraph_with_format_to(doc_desc, act_instructions)

        # Filtrar el cronograma para la actividad (usando la columna "Descripcion de la actividad")
        if "Descripcion de la actividad" in cronograma_unit.columns:
            filtro_act = cronograma_unit["Descripcion de la actividad"].astype(str).str.strip().str.contains(
                activity_title, case=False
            )
            cronograma_act = cronograma_unit.loc[filtro_act]
        else:
            print(f"Advertencia: La columna 'Descripcion de la actividad' no se encontró para la Unidad {unit_num}.")
            cronograma_act = cronograma_unit

        # Insertar tabla con la primera fila encontrada en el cronograma para la actividad
        if not cronograma_act.empty:
            cronograma_act = cronograma_act.head(1)
            table = doc_desc.add_table(rows=1, cols=len(cronograma_act.columns))
            table.autofit = True

            hdr_cells = table.rows[0].cells
            for j, col_name in enumerate(cronograma_act.columns):
                hdr_cells[j].text = col_name
                run = hdr_cells[j].paragraphs[0].runs[0]
                run.font.bold = True
                run.font.size = Pt(9)
                run.font.color.rgb = RGBColor(255, 255, 255)
                shading_elm = parse_xml(r'<w:shd {} w:fill="654321"/>'.format(nsdecls('w')))
                hdr_cells[j]._element.get_or_add_tcPr().append(shading_elm)

            for idx, row in cronograma_act.iterrows():
                cells = table.add_row().cells
                for j, value in enumerate(row):
                    text_value = value.strftime("%d/%m/%Y") if isinstance(value, datetime) else str(value)
                    cells[j].text = text_value
                    for paragraph in cells[j].paragraphs:
                        for run in paragraph.runs:
                            run.font.size = Pt(9)
                fill_color = "FFEFD5" if idx % 2 == 0 else "FFFFFF"
                for cell in cells:
                    cell_shading = parse_xml(r'<w:shd {} w:fill="{}"/>'.format(nsdecls('w'), fill_color))
                    cell._element.get_or_add_tcPr().append(cell_shading)

            tbl = table._tbl
            tblPr = tbl.tblPr
            tblBorders = parse_xml(
                r'<w:tblBorders {}>'
                r'<w:top w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
                r'<w:left w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
                r'<w:bottom w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
                r'<w:right w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
                r'<w:insideH w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
                r'<w:insideV w:val="single" w:sz="4" w:space="0" w:color="808080"/>'
                r'</w:tblBorders>'.format(nsdecls('w'))
            )
            tblPr.append(tblBorders)
            auto_adjust_column_widths(table, cronograma_act)
        else:
            add_paragraph_with_format_to(doc_desc, "No se encontraron datos en el cronograma para esta actividad de la unidad.")

        add_paragraph_with_format_to(
            doc_desc,
            "Referencias: AA: Actividad de aprendizaje individual, AI: Actividad Interactiva, AF: Actividad de Fijación"
        )
        add_paragraph_with_format_to(
            doc_desc,
            "Ante cualquier duda, recuerda que puedes manifestarla durante las tutorías sincrónicas o en el foro Consultas de esta unidad en el aula virtual. Estoy aquí para acompañarte."
        )

        output_desc = f"{output_desc_base}_Unidad{unit_num}_Act{act_num}.docx"
        doc_desc.save(output_desc)
        print(f"Documento (Word) de la Actividad {unit_num}.{act_num} generado en: {output_desc}")

        try:
            output_pdf = f"{output_desc_base}_Unidad{unit_num}_Act{act_num}.pdf"
            convert(output_desc, output_pdf)
            print(f"Documento (PDF) de la Actividad {unit_num}.{act_num} generado en: {output_pdf}\n")
        except Exception as e:
            print(f"[Error] No se pudo convertir a PDF la Actividad {unit_num}.{act_num}: {str(e)}\n")


Documento (Word) de la Actividad 1.1 generado en: G:/Mi unidad/FACEN_BIGDATA/Descripcion_Actividad_Unidad1_Act1.docx


  0%|          | 0/1 [00:00<?, ?it/s]

Documento (PDF) de la Actividad 1.1 generado en: G:/Mi unidad/FACEN_BIGDATA/Descripcion_Actividad_Unidad1_Act1.pdf

Documento (Word) de la Actividad 1.2 generado en: G:/Mi unidad/FACEN_BIGDATA/Descripcion_Actividad_Unidad1_Act2.docx


  0%|          | 0/1 [00:00<?, ?it/s]

Documento (PDF) de la Actividad 1.2 generado en: G:/Mi unidad/FACEN_BIGDATA/Descripcion_Actividad_Unidad1_Act2.pdf

Documento (Word) de la Actividad 1.3 generado en: G:/Mi unidad/FACEN_BIGDATA/Descripcion_Actividad_Unidad1_Act3.docx


...

Documento (PDF) de la Actividad 5.1 generado en: G:/Mi unidad/FACEN_BIGDATA/Descripcion_Actividad_Unidad5_Act1.pdf

Documento (Word) de la Actividad 5.2 generado en: G:/Mi unidad/FACEN_BIGDATA/Descripcion_Actividad_Unidad5_Act2.docx


  0%|          | 0/1 [00:00<?, ?it/s]

Documento (PDF) de la Actividad 5.2 generado en: G:/Mi unidad/FACEN_BIGDATA/Descripcion_Actividad_Unidad5_Act2.pdf

Documento (Word) de la Actividad 5.3 generado en: G:/Mi unidad/FACEN_BIGDATA/Descripcion_Actividad_Unidad5_Act3.docx


  0%|          | 0/1 [00:00<?, ?it/s]

Documento (PDF) de la Actividad 5.3 generado en: G:/Mi unidad/FACEN_BIGDATA/Descripcion_Actividad_Unidad5_Act3.pdf



# VISTAS PARA UNIDADES


In [24]:
from PIL import Image, ImageDraw, ImageFont
import os

# Información de cada unidad
units_info = [
    {
        "title": "Analítica de Big Data",
        "unit": "Unidad 1:\nManejo de datos en Big Data\nAlgunas alternativas",
        "background": "G:/Mi unidad/FACEN_BIGDATA/fondo_vista_unidades.png",
        "output": "G:/Mi unidad/FACEN_BIGDATA/portada_unidad1.png"
    },
    {
        "title": "Analítica de Big Data",
        "unit": "Unidad 2:\nExploración y visualización datos en Big Data",
        "background": "G:/Mi unidad/FACEN_BIGDATA/fondo_vista_unidades.png",
        "output": "G:/Mi unidad/FACEN_BIGDATA/portada_unidad2.png"
    },
    {
        "title": "Analítica de Big Data",
        "unit": "Unidad 3:\nMachine Learning",
        "background": "G:/Mi unidad/FACEN_BIGDATA/fondo_vista_unidades.png",
        "output": "G:/Mi unidad/FACEN_BIGDATA/portada_unidad3.png"
    },
    {
        "title": "Analítica de Big Data",
        "unit": "Unidad 4:\nInterpretación de los resultados\ny modelos estadísticos",
        "background": "G:/Mi unidad/FACEN_BIGDATA/fondo_vista_unidades.png",
        "output": "G:/Mi unidad/FACEN_BIGDATA/portada_unidad4.png"
    },
    {
        "title": "Analítica de Big Data",
        "unit": "Unidad 5:\nTrabajo práctico final",
        "background": "G:/Mi unidad/FACEN_BIGDATA/fondo_vista_unidades.png",
        "output": "G:/Mi unidad/FACEN_BIGDATA/portada_unidad5.png"
    }
]

# Configuración de fuentes
try:
    # Cargar fuentes personalizadas (o usar fuentes predeterminadas)
    font_large = ImageFont.truetype("arial.ttf", 60)
    font_medium = ImageFont.truetype("arial.ttf", 45)
    font_small = ImageFont.truetype("arial.ttf", 30)
except IOError:
    # Usar fuentes básicas si no se encuentran las fuentes personalizadas
    font_large = font_medium = font_small = ImageFont.load_default()

def create_unit_cover_improved(unit_data):
    try:
        # Cargar imagen de fondo
        background_path = unit_data.get("background", "")
        if not os.path.exists(background_path):
            raise FileNotFoundError(f"Imagen de fondo no encontrada: {background_path}")

        img = Image.open(background_path)
        draw = ImageDraw.Draw(img)

        # Dimensiones de la imagen
        img_width, img_height = img.size

        # Coordenadas para centrar el texto en el lado derecho
        x_center = img_width//2  # Centrado en el lado derecho de la imagen
        y_offset = img_height//3.5
    

        # Extraer datos
        title = unit_data.get("title", "")
        unit = unit_data.get("unit", "")

        # Dibujar el texto
        draw.text((x_center, y_offset), title, font=font_large, fill="orange", anchor="mm")
        draw.multiline_text(
            (x_center, y_offset + 150),
            unit,
            font=font_medium,
            fill="black",
            anchor="mm",
            align="center"
        )

        # Guardar imagen generada
        output_path = unit_data.get("output", "output_improved.png")
        img.save(output_path)

        return output_path

    except Exception as e:
        return f"Error al crear portada mejorada: {str(e)}"

# Crear las portadas mejoradas con manejo seguro
improved_outputs = [create_unit_cover_improved(unit) for unit in units_info]
improved_outputs


['G:/Mi unidad/FACEN_BIGDATA/portada_unidad1.png',
 'G:/Mi unidad/FACEN_BIGDATA/portada_unidad2.png',
 'G:/Mi unidad/FACEN_BIGDATA/portada_unidad3.png',
 'G:/Mi unidad/FACEN_BIGDATA/portada_unidad4.png',
 'G:/Mi unidad/FACEN_BIGDATA/portada_unidad5.png']

# MATERIALES DE LECTURA

In [ ]:
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx2pdf import convert

# -----------------------------------------------------------------
# Rutas de archivos
# -----------------------------------------------------------------
plantilla_path = "G:/Mi unidad/FACEN_BIGDATA/BigData_plantilla_Guía_Didáctica.docx"
output_word_path = "G:/Mi unidad/FACEN_BIGDATA/Material_Lectura_Unidad1_Extendido.docx"
output_pdf_path = "G:/Mi unidad/FACEN_BIGDATA/Material_Lectura_Unidad1_Extendido.pdf"

# -----------------------------------------------------------------
# Función auxiliar para agregar párrafos con formato
# -----------------------------------------------------------------
def add_paragraph_with_format(doc, text, style=None):
    p = doc.add_paragraph(text, style=style)
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    p.paragraph_format.first_line_indent = Inches(0.5)
    return p

# -----------------------------------------------------------------
# 1. Crear documento a partir de la plantilla
# -----------------------------------------------------------------
doc = Document(plantilla_path)

# -----------------------------------------------------------------
# 2. Portada adaptada
# -----------------------------------------------------------------
cover_texts = [
    "", "", 
    "Departamento de Educación a Distancia",
    "", "", "",
    "Analítica de Big Data",
    "", "", "", "", "", "", "", "", "", "", ""
]

cover_texts2 = ["Año 2025"]

for text in cover_texts:
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.size = Pt(38)
    run.font.color.rgb = RGBColor(102, 51, 0)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER

material_run = doc.add_paragraph().add_run("Material de Lectura - Unidad 1")
material_run.font.size = Pt(32)
material_run.font.color.rgb = RGBColor(0, 0, 0)
doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER

for text in cover_texts2:
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.size = Pt(28)
    run.font.color.rgb = RGBColor(0, 0, 0)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc.add_page_break()

# -----------------------------------------------------------------
# 3. Contenido principal extendido para la Unidad 1
# -----------------------------------------------------------------
doc.add_heading("Unidad 1: Introducción al manejo de datos en Big Data", level=1)

add_paragraph_with_format(
    doc,
    "El concepto de Big Data ha emergido como uno de los pilares fundamentales de la sociedad digital. Hoy en día, las organizaciones "
    "pueden recopilar, almacenar y analizar enormes volúmenes de datos para mejorar procesos, reducir costos y optimizar la toma de decisiones. "
    "La evolución de la tecnología ha permitido que diversos sectores —como la salud, la educación, la industria y el comercio electrónico— "
    "saquen provecho de estos grandes volúmenes de información. En esta unidad, exploraremos los orígenes del Big Data, los cambios de "
    "paradigma en el análisis de datos y sus principales características, así como ejemplos prácticos que ilustran su aplicación."
)

doc.add_heading("1.1 Orígenes del Big Data", level=2)
add_paragraph_with_format(
    doc,
    "El término 'Big Data' se popularizó con el auge de la World Wide Web y la necesidad de procesar e interpretar la ingente cantidad "
    "de información que se generaba en línea. Sin embargo, sus raíces se remontan a disciplinas científicas que, desde hace décadas, "
    "manejan grandes volúmenes de datos. Ejemplos de ello son la astronomía, con la recopilación de información sobre millones de estrellas "
    "y galaxias, y la genética, con proyectos como el Genoma Humano, que generaba terabytes de datos genéticos que requerían nuevas estrategias "
    "de almacenamiento y análisis."
)

add_paragraph_with_format(
    doc,
    "En la década de 2000, empresas como Google y Yahoo! enfrentaron retos similares al indexar y procesar datos de la web, lo que propició "
    "el desarrollo de tecnologías clave como MapReduce, Hadoop y HDFS (Hadoop Distributed File System). Estos sistemas distribuían el trabajo "
    "de procesamiento en múltiples nodos, permitiendo manejar grandes cantidades de información de forma más eficiente y escalable. "
    "A partir de entonces, el término Big Data se consolidó como un enfoque tecnológico y de análisis de datos a gran escala."
)

doc.add_heading("1.2 Cambio de paradigma en el análisis de datos", level=2)
add_paragraph_with_format(
    doc,
    "El cambio de paradigma en el análisis de datos se observa en la transición de un enfoque basado en muestras y en la búsqueda de "
    "causalidades explícitas, a un enfoque donde el volumen masivo de información permite trabajar con correlaciones y patrones "
    "sin necesidad de hipótesis previas. Esto no significa que la causalidad deje de ser importante, sino que en contextos masivos "
    "es posible descubrir relaciones entre variables que habrían pasado inadvertidas con métodos estadísticos tradicionales."
)

add_paragraph_with_format(
    doc,
    "Un ejemplo icónico de este cambio lo constituye Google Flu Trends, el cual analizó datos de búsquedas relacionadas con la gripe "
    "para predecir brotes epidémicos. Aunque en la práctica enfrentó desafíos de precisión, demostró el potencial de analizar conjuntos "
    "masivos de datos en tiempo real. Este giro hacia el 'todo el dato' —en lugar de solo muestras— se ha potenciado con la creciente "
    "capacidad de cómputo y el abaratamiento del almacenamiento digital."
)

doc.add_heading("1.3 Las 4 V del Big Data", level=2)

doc.add_heading("Volumen", level=3)
add_paragraph_with_format(
    doc,
    "El volumen hace referencia a la cantidad masiva de datos que se generan y recopilan. Se estima que cada día se crean más de "
    "2.5 exabytes de datos a nivel mundial. Esta magnitud es tal que, en algunos casos, no es factible almacenarlos en sistemas "
    "convencionales. De esta necesidad surge el desarrollo de arquitecturas distribuidas y soluciones de almacenamiento escalables, "
    "como los data lakes o los sistemas de archivos distribuidos (HDFS)."
)

doc.add_heading("Velocidad", level=3)
add_paragraph_with_format(
    doc,
    "La velocidad alude a la rapidez con la que se generan y se deben procesar los datos. Cada vez más aplicaciones requieren análisis "
    "en tiempo real o casi en tiempo real (near real-time), como en plataformas de trading financiero o sistemas de detección de fraude. "
    "Tecnologías como Apache Kafka y Apache Spark son muy empleadas para procesar flujos de datos (streams) que llegan de forma continua."
)

doc.add_heading("Variedad", level=3)
add_paragraph_with_format(
    doc,
    "La variedad se relaciona con la diversidad de fuentes y formatos que componen el Big Data. Hoy día, no solo se trabaja con datos "
    "estructurados (p. ej., bases de datos relacionales), sino también con datos semiestructurados (JSON, XML) y datos no estructurados "
    "(texto libre, imágenes, videos, logs de servidores, redes sociales). Integrar estas diferentes tipologías exige un procesamiento "
    "y un modelado que trascienden los métodos convencionales de bases de datos."
)

doc.add_heading("Veracidad", level=3)
add_paragraph_with_format(
    doc,
    "La veracidad se refiere a la calidad, exactitud y confiabilidad de los datos. Grandes volúmenes de información pueden contener "
    "errores, duplicados o sesgos que distorsionan los resultados de los análisis. Por ello, resultan esenciales las tareas de "
    "limpieza, validación y aseguramiento de la calidad de los datos (Data Quality)."
)

doc.add_heading("1.4 Gestión de datos en Big Data", level=2)
add_paragraph_with_format(
    doc,
    "La gestión de datos en un entorno de Big Data implica abordar varias etapas y procesos, entre los que se incluyen la extracción, "
    "transformación y carga (ETL), el almacenamiento distribuido y el análisis paralelo. Entre las tecnologías destacadas se encuentran:"
)
add_paragraph_with_format(
    doc,
    "• Apache Hadoop: Uno de los primeros ecosistemas de Big Data, conformado por HDFS (almacenamiento distribuido), YARN (gestión de "
    "recursos) y MapReduce (paradigma de programación para el procesamiento de datos)."
)
add_paragraph_with_format(
    doc,
    "• Apache Spark: Una herramienta que permite el procesamiento distribuido de datos en memoria, superando a MapReduce en velocidad "
    "en muchos casos y facilitando la ejecución de algoritmos de machine learning, streaming y SQL en un mismo entorno."
)
add_paragraph_with_format(
    doc,
    "• Bases de datos NoSQL: Diseñadas para manejar datos de alta velocidad y variedad. Algunas de las más populares incluyen MongoDB "
    "(documentos), Cassandra (columnas), Redis (claves-valor) y Neo4j (grafos)."
)

doc.add_heading("1.5 Ejemplos de aplicación del Big Data", level=2)

doc.add_heading("Marketing y comportamiento del cliente", level=3)
add_paragraph_with_format(
    doc,
    "Las organizaciones utilizan datos de navegación web, historiales de compras, interacciones en redes sociales y datos demográficos "
    "para crear perfiles de clientes y personalizar sus ofertas. Netflix, por ejemplo, combina la información de millones de usuarios "
    "para ofrecer recomendaciones precisas de películas y series, impactando positivamente en la retención de clientes."
)

add_paragraph_with_format(
    doc,
    "Otro caso es Amazon, que emplea algoritmos de filtrado colaborativo para sugerir productos basados en las búsquedas y compras de "
    "otros usuarios con patrones de consumo similares. Estas estrategias de recomendación personalizadas mejoran la tasa de conversión "
    "y aumentan los ingresos."
)

doc.add_heading("Salud y biotecnología", level=3)
add_paragraph_with_format(
    doc,
    "En el ámbito de la salud, Big Data se emplea para el análisis de registros médicos electrónicos (EMR), monitorización de pacientes "
    "en tiempo real, investigación genética y descubrimiento de fármacos. El análisis de datos de pacientes, junto con información "
    "genómica, permite identificar enfermedades con mayor precisión y diseñar tratamientos personalizados. Asimismo, algoritmos de "
    "machine learning son capaces de detectar patrones en imágenes médicas —radiografías, resonancias magnéticas— para diagnosticar "
    "enfermedades de forma temprana."
)

doc.add_heading("Logística y optimización de rutas", level=3)
add_paragraph_with_format(
    doc,
    "Las compañías de transporte y logística, como UPS o DHL, utilizan grandes volúmenes de datos para optimizar rutas de entrega, "
    "considerando factores como tráfico en tiempo real, condiciones climáticas y distribución geográfica de los clientes. Esto se "
    "traduce en una reducción de costos, menores emisiones de CO2 y entregas más rápidas."
)

doc.add_heading("Sector financiero y detección de fraudes", level=3)
add_paragraph_with_format(
    doc,
    "Los bancos y entidades financieras aplican técnicas analíticas en tiempo real para detectar comportamientos sospechosos en las "
    "transacciones de los clientes. El uso de Big Data, combinado con algoritmos de inteligencia artificial, permite evaluar patrones "
    "de compra, ubicación geográfica y perfiles de riesgo, emitiendo alertas tempranas de actividad fraudulenta."
)

doc.add_heading("1.6 Desafíos y tendencias actuales del Big Data", level=2)
add_paragraph_with_format(
    doc,
    "A pesar de sus numerosas ventajas, el Big Data plantea desafíos que las organizaciones deben gestionar adecuadamente:"
)
add_paragraph_with_format(
    doc,
    "• Privacidad y protección de datos: La recopilación masiva de información personal ha generado un debate ético y legal. "
    "En muchos países, se han implementado regulaciones como el RGPD (Reglamento General de Protección de Datos en la Unión Europea) "
    "para garantizar la privacidad de los ciudadanos."
)
add_paragraph_with_format(
    doc,
    "• Escalabilidad y costo: Procesar volúmenes cada vez mayores de datos implica la inversión en infraestructura y la adopción "
    "de soluciones en la nube, lo cual conlleva costos de almacenamiento y cómputo que pueden ser elevados si no se gestionan "
    "eficientemente."
)
add_paragraph_with_format(
    doc,
    "• Falta de talento especializado: El mercado demanda profesionales con habilidades en análisis de datos, estadística, "
    "machine learning y plataformas de Big Data. La escasez de expertos en estas áreas dificulta la adopción de estrategias "
    "de Big Data en algunas organizaciones."
)
add_paragraph_with_format(
    doc,
    "En cuanto a tendencias, destacan los avances en arquitecturas de procesamiento en tiempo real (streaming) y la aparición de "
    "modelos de Inteligencia Artificial generativa, que demandan grandes cantidades de datos para su entrenamiento y despliegue. "
    "Asimismo, la creciente adopción de arquitecturas de microservicios y contenedores (Docker, Kubernetes) está facilitando la "
    "implementación y la escalabilidad de soluciones de Big Data de manera más ágil."
)

doc.add_heading("1.7 Herramientas y ecosistemas de Big Data", level=2)
add_paragraph_with_format(
    doc,
    "Además de las soluciones clásicas como Hadoop y Spark, existen ecosistemas cada vez más ricos para trabajar con Big Data:"
)
add_paragraph_with_format(
    doc,
    "• Herramientas de ingesta y streaming: Apache Kafka, Apache Flume, AWS Kinesis y Azure Event Hubs se utilizan para capturar "
    "y procesar flujos de datos que se generan de forma continua."
)
add_paragraph_with_format(
    doc,
    "• Sistemas de almacenamiento en la nube: Plataformas como Amazon S3, Google Cloud Storage y Azure Data Lake Storage ofrecen "
    "espacios escalables para almacenar petabytes de datos con alta disponibilidad."
)
add_paragraph_with_format(
    doc,
    "• Librerías de análisis y machine learning: Dentro de Spark, librerías como MLlib permiten entrenar modelos de aprendizaje "
    "automático de forma distribuida. Otras librerías, como TensorFlow y PyTorch, pueden integrarse con clústeres de Big Data para "
    "gestionar grandes volúmenes de datos en entornos de Deep Learning."
)

doc.add_heading("1.8 Futuro del Big Data y conclusiones de la Unidad 1", level=2)
add_paragraph_with_format(
    doc,
    "El futuro del Big Data apunta hacia una mayor automatización en todas las fases de la gestión de datos, desde la ingesta hasta "
    "el análisis y la presentación de resultados. Las técnicas de Inteligencia Artificial, en especial el aprendizaje profundo, "
    "seguirán impulsando el desarrollo de soluciones para procesar de forma inteligente toda esta información."
)
add_paragraph_with_format(
    doc,
    "Asimismo, el auge del Internet de las Cosas (IoT) generará aún más datos provenientes de sensores en ciudades, fábricas y hogares "
    "inteligentes, lo que aumentará la necesidad de arquitecturas de Big Data capaces de procesar información de manera distribuida "
    "y en tiempo real."
)
add_paragraph_with_format(
    doc,
    "En conclusión, el Big Data se ha convertido en un habilitador clave para la competitividad en la economía digital. Su correcta "
    "implementación no solo permite mejorar la eficiencia operativa y la toma de decisiones basada en datos, sino también promover "
    "la innovación en ámbitos tan diversos como la salud, la logística, el comercio y la ciencia."
)

doc.add_heading("Bibliografía", level=1)
add_paragraph_with_format(
    doc,
    "• Conesa i Caralt, J., & Gómez García, J. L. (2025). Introducción al Big Data."
)
add_paragraph_with_format(
    doc,
    "• IBM. (s.f.). Las 4 V del Big Data. Disponible en: http://www.ibmbigdatahub.com"
)
add_paragraph_with_format(
    doc,
    "• Sloan Digital Sky Survey. Disponible en: http://www.sdss.org/"
)
add_paragraph_with_format(
    doc,
    "• Google Flu Trends. Disponible en: http://www.google.org/flutrends/"
)
add_paragraph_with_format(
    doc,
    "• Marr, B. (2016). Big Data: Using SMART Big Data, Analytics and Metrics To Make Better Decisions and Improve Performance. John Wiley & Sons."
)
add_paragraph_with_format(
    doc,
    "• Chen, M., Mao, S., & Liu, Y. (2014). Big data: A survey. Mobile Networks and Applications, 19(2), 171-209."
)

# -----------------------------------------------------------------
# 4. Guardar el documento Word
# -----------------------------------------------------------------
doc.save(output_word_path)
print(f"Documento Word generado exitosamente en: {output_word_path}")

# -----------------------------------------------------------------
# 5. Convertir a PDF usando docx2pdf
# -----------------------------------------------------------------
try:
    convert(input_path=output_word_path, output_path=output_pdf_path)
    print(f"Documento PDF generado exitosamente en: {output_pdf_path}")
except Exception as e:
    print("Ocurrió un error al convertir a PDF:", str(e))


Documento Word generado exitosamente en: G:/Mi unidad/FACEN_BIGDATA/Material_Lectura_Unidad1_Extendido.docx


  0%|          | 0/1 [00:00<?, ?it/s]

Documento PDF generado exitosamente en: G:/Mi unidad/FACEN_BIGDATA/Material_Lectura_Unidad1_Extendido.pdf


## Material de lectura - Unidad 2

In [ ]:
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx2pdf import convert

# Para sombrear celdas (en bloques de código)
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# -----------------------------------------------------------------
# Rutas de archivos
# -----------------------------------------------------------------
plantilla_path = "G:/Mi unidad/FACEN_BIGDATA/BigData_plantilla_Guía_Didáctica.docx"
output_word_path = "G:/Mi unidad/FACEN_BIGDATA/Material_Lectura_Unidad2_Extendido.docx"
output_pdf_path = "G:/Mi unidad/FACEN_BIGDATA/Material_Lectura_Unidad2_Extendido.pdf"

# -----------------------------------------------------------------
# Funciones auxiliares
# -----------------------------------------------------------------
def add_paragraph_with_format(doc, text):
    """
    Crea un párrafo con texto justificado e indentación de primera línea.
    """
    p = doc.add_paragraph(text)
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    p.paragraph_format.first_line_indent = Inches(0.5)
    return p

def add_list_item(doc, text):
    """
    Crea un párrafo normal (sin estilo de lista específico),
    y añade el texto como ítem de la lista.
    """
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    p.add_run(text)

def add_code_block(doc, code_text, bg_color="EEE8D5"):
    """
    Inserta un recuadro (tabla de 1x1) con fondo color arena (EEE8D5) 
    para resaltar bloques de código.
    """
    table = doc.add_table(rows=1, cols=1)
    table.allow_autofit = True
    table.style = 'Table Grid'  # Ajusta si deseas otro estilo de tabla

    # Sombreado de la celda
    cell = table.cell(0, 0)
    shading_elm = OxmlElement('w:shd')
    shading_elm.set(qn('w:val'), 'clear')
    shading_elm.set(qn('w:color'), 'auto')
    shading_elm.set(qn('w:fill'), bg_color)
    cell._tc.get_or_add_tcPr().append(shading_elm)

    # Añadimos el código en la celda con fuente monoespaciada
    p = cell.add_paragraph()
    run = p.add_run(code_text)
    run.font.name = 'Consolas'
    run.font.size = Pt(9)

    # Espacio tras el bloque
    doc.add_paragraph()

# -----------------------------------------------------------------
# 1. Crear documento a partir de la plantilla
# -----------------------------------------------------------------
doc = Document(plantilla_path)

# -----------------------------------------------------------------
# 2. Portada adaptada para la Unidad 2
# -----------------------------------------------------------------
cover_texts = [
    "", "", 
    "Curso de Actualización",
    "", "", "",
    "Introducción a la Analítica de Big Data",
    "", "", "", "", "", "", "", "", "", "", ""
]

cover_texts2 = ["Año 2025"]

for text in cover_texts:
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.size = Pt(38)
    run.font.color.rgb = RGBColor(102, 51, 0)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER

material_run = doc.add_paragraph().add_run("Material de Lectura - Unidad 2")
material_run.font.size = Pt(32)
material_run.font.color.rgb = RGBColor(0, 0, 0)
doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER

for text in cover_texts2:
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.size = Pt(28)
    run.font.color.rgb = RGBColor(0, 0, 0)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc.add_page_break()

# -----------------------------------------------------------------
# 3. Contenido principal para la Unidad 2
# -----------------------------------------------------------------
doc.add_heading("Unidad 2: Visualización y exploración de datos en Big Data", level=1)
add_paragraph_with_format(
    doc,
    "La visualización y exploración de datos (también conocida como Análisis Exploratorio de Datos, EDA) "
    "es fundamental en cualquier proyecto de Big Data o ciencia de datos. En este bloque, aprenderemos "
    "a identificar la naturaleza de las variables, sus distribuciones, así como diversas técnicas de "
    "preprocesado e imputación de valores ausentes. También se mostrarán ejemplos en Python y R para "
    "reforzar los conceptos teóricos."
)

# Sección 4.2.1
doc.add_heading("4.2.1 Tipos de variables y distribución", level=2)
add_paragraph_with_format(
    doc,
    "En un entorno de Big Data, es clave reconocer los tipos de variables y entender sus distribuciones, "
    "pues esto influirá en la elección de técnicas estadísticas y de visualización. Los principales tipos "
    "de variables son:"
)
add_list_item(doc, "Cuantitativas: pueden ser discretas (número de clientes) o continuas (tiempo, peso).")
add_list_item(doc, "Cualitativas: nominales (sin orden) u ordinales (con un orden inherente).")

add_paragraph_with_format(
    doc,
    "Por otro lado, la distribución (normal, sesgada, multimodal, etc.) describe cómo se concentran los valores "
    "en su rango y determina si se requieren transformaciones (log, Box-Cox, etc.) para su análisis."
)

# Sección 4.2.2
doc.add_heading("4.2.2 Exploración de datos", level=2)
add_paragraph_with_format(
    doc,
    "La exploración de datos o EDA consiste en un conjunto de técnicas destinadas a entender la estructura "
    "y características básicas del conjunto de datos antes de aplicar modelos más complejos. Algunas tareas "
    "típicas incluyen:"
)
add_list_item(doc, "Cálculo de estadísticas descriptivas: media, mediana, desviación estándar.")
add_list_item(doc, "Visualizaciones univariadas: histogramas, boxplots.")
add_list_item(doc, "Visualizaciones multivariadas: scatter plots, mapas de calor de correlaciones.")

# Ejemplo en Python: Exploración de datos
doc.add_heading("Ejemplo en Python: Exploración de datos", level=3)
add_paragraph_with_format(
    doc,
    "A continuación, se muestra un script básico en Python, que emplea las librerías pandas, matplotlib y seaborn. "
    "Realiza las siguientes acciones:"
)

# Ítems enumerados
add_list_item(doc, "Cargar un dataset en formato CSV.")
add_list_item(doc, "Mostrar estadísticas descriptivas.")
add_list_item(doc, "Graficar la distribución de una variable cuantitativa.")
add_list_item(doc, "Visualizar la correlación entre variables.")

python_code_example = (
    "import pandas as pd\n"
    "import matplotlib.pyplot as plt\n"
    "import seaborn as sns\n\n"
    "# 1. Cargamos el dataset\n"
    "df = pd.read_csv('clientes_compras.csv')\n\n"
    "# 2. Observamos las primeras filas\n"
    "print(df.head())\n\n"
    "# 3. Resumen estadístico de las variables numéricas\n"
    "print(df.describe())\n\n"
    "# 4. Visualización de la distribución de la variable 'edad'\n"
    "sns.histplot(data=df, x='edad', kde=True)\n"
    "plt.title('Distribución de la edad de los clientes')\n"
    "plt.show()\n\n"
    "# 5. Matriz de correlaciones\n"
    "corr = df.corr()\n"
    "sns.heatmap(corr, annot=True, cmap='coolwarm')\n"
    "plt.title('Mapa de calor de correlaciones')\n"
    "plt.show()\n"
)
add_code_block(doc, code_text=python_code_example)

add_paragraph_with_format(
    doc,
    "En este ejemplo, `read_csv` de pandas permite cargar datos de un archivo CSV. Con `describe()` obtenemos "
    "estadísticas como media, percentiles y desviación estándar. Por su parte, `histplot` y `heatmap` facilitan "
    "la visualización de la distribución de una variable y la relación entre variables, respectivamente."
)

# Ejemplo en R: Exploración de datos
doc.add_heading("Ejemplo en R: Exploración de datos", level=3)
add_paragraph_with_format(
    doc,
    "De manera análoga, en R podemos usar ggplot2 para crear gráficos, y funciones básicas para la exploración "
    "preliminar. En el siguiente código se ejemplifican los mismos pasos:"
)

r_code_example = (
    "# Instalamos y cargamos las librerías necesarias\n"
    "# install.packages('ggplot2')\n"
    "library(ggplot2)\n\n"
    "# 1. Cargamos el dataset (ej., un archivo CSV)\n"
    "df <- read.csv('clientes_compras.csv', header=TRUE)\n\n"
    "# 2. Observamos las primeras filas\n"
    "head(df)\n\n"
    "# 3. Resumen estadístico\n"
    "summary(df)\n\n"
    "# 4. Histograma de la edad\n"
    "ggplot(df, aes(x=edad)) +\n"
    "  geom_histogram(binwidth=5, fill='blue', color='white') +\n"
    "  labs(title='Distribución de la edad de los clientes', x='Edad', y='Frecuencia')\n\n"
    "# 5. (Opcional) Gráfico de correlaciones con GGally\n"
    "# install.packages('GGally')\n"
    "library(GGally)\n"
    "ggcorr(df, label=TRUE)\n"
)
add_code_block(doc, code_text=r_code_example)

add_paragraph_with_format(
    doc,
    "La función `summary()` devuelve medidas como mínimo, cuartiles, media y máximo, mientras que `geom_histogram()` "
    "permite visualizar de forma sencilla la distribución de una variable. Además, la librería `GGally` ofrece "
    "funciones como `ggcorr` para matrices de correlaciones."
)

# Sección 4.2.3
doc.add_heading("4.2.3 Preprocesado de los datos", level=2)
add_paragraph_with_format(
    doc,
    "El preprocesado consiste en limpiar y transformar los datos para asegurar que estén completos y coherentes "
    "antes de aplicarse modelos predictivos. Suele incluir las siguientes etapas:"
)
add_list_item(doc, "Limpieza de datos: eliminar duplicados, manejar valores atípicos.")
add_list_item(doc, "Transformación de variables: normalización o estandarización para mejorar la comparabilidad.")
add_list_item(doc, "Selección de características: reducir la dimensionalidad conservando solo variables relevantes.")

# Sección 4.2.4
doc.add_heading("4.2.4 Imputación de valores ausentes", level=2)
add_paragraph_with_format(
    doc,
    "En el mundo real, muchos datasets contienen valores faltantes (missing values). El modo de manejarlos puede "
    "influir enormemente en los resultados del análisis. Entre las estrategias comunes encontramos:"
)
add_list_item(doc, "Eliminación de registros con datos faltantes (si la proporción es muy baja).")
add_list_item(doc, "Imputación con estadísticos (media, mediana, moda).")
add_list_item(doc, "Modelos predictivos para estimar el valor faltante, como KNN o regresiones múltiples.")

# Ejemplo de imputación en Python
doc.add_heading("Ejemplo en Python: Imputación de valores ausentes", level=3)
add_paragraph_with_format(
    doc,
    "A continuación, se muestra un ejemplo para manejar valores ausentes: eliminarlos, o imputarlos con la media o "
    "la mediana."
)

python_imputation_example = (
    "import pandas as pd\n\n"
    "# Cargamos el dataset con valores faltantes\n"
    "df = pd.read_csv('datos_incompletos.csv')\n\n"
    "# Opción 1: Eliminar filas con valores ausentes\n"
    "df_drop = df.dropna()\n\n"
    "# Opción 2: Imputar con la media (para variables numéricas)\n"
    "df_fill_mean = df.fillna(df.mean())\n\n"
    "# Opción 3: Imputar con la mediana\n"
    "df_fill_median = df.fillna(df.median())\n\n"
    "# Verificamos cuántos valores faltantes quedan en cada caso\n"
    "print('Valores ausentes tras eliminar filas:', df_drop.isna().sum())\n"
    "print('Valores ausentes tras imputar con media:', df_fill_mean.isna().sum())\n"
    "print('Valores ausentes tras imputar con mediana:', df_fill_median.isna().sum())\n"
)
add_code_block(doc, code_text=python_imputation_example)

add_paragraph_with_format(
    doc,
    "Si la cantidad de datos ausentes es muy pequeña y se distribuye aleatoriamente, `dropna()` puede ser útil. "
    "De lo contrario, la imputación con estadísticos (media, mediana) es un primer paso sencillo pero que podría "
    "introducir sesgos. Para Big Data, se recomiendan métodos más robustos, como KNNImputer o algoritmos de ML."
)

# Ejemplo de imputación en R
doc.add_heading("Ejemplo en R: Imputación de valores ausentes", level=3)
add_paragraph_with_format(
    doc,
    "En R, igualmente podemos optar por eliminar filas incompletas con `na.omit()` o realizar imputaciones "
    "con funciones como `mean()` o `median()`. El siguiente ejemplo ilustra estas opciones:"
)

r_imputation_example = (
    "# Cargamos el dataset con valores faltantes\n"
    "df <- read.csv('datos_incompletos.csv')\n\n"
    "# Opción 1: Eliminar filas con NA\n"
    "df_drop <- na.omit(df)\n\n"
    "# Opción 2: Imputar con la media para una columna numérica\n"
    "df$col_numerica[is.na(df$col_numerica)] <- mean(df$col_numerica, na.rm = TRUE)\n\n"
    "# Opción 3: Imputar con la mediana\n"
    "df$col_numerica[is.na(df$col_numerica)] <- median(df$col_numerica, na.rm = TRUE)\n\n"
    "# Verificamos valores ausentes restantes\n"
    "sum(is.na(df))\n"
)
add_code_block(doc, code_text=r_imputation_example)

add_paragraph_with_format(
    doc,
    "Además de estos métodos básicos, R ofrece paquetes más avanzados como `mice` (Multiple Imputation by Chained Equations), "
    "que implementa métodos de imputación múltiple para estimar valores faltantes de forma más precisa."
)

# Conclusión de la Unidad 2
doc.add_heading("Conclusiones de la Unidad 2", level=2)
add_paragraph_with_format(
    doc,
    "A lo largo de esta unidad, hemos revisado la importancia de la exploración y visualización de datos (EDA), los tipos y "
    "distribuciones de variables, las fases de preprocesado y las técnicas de imputación de valores ausentes. Estas prácticas "
    "garantizan la calidad y la coherencia de la información antes de construir modelos complejos. Con ejemplos en Python y R, "
    "se han sentado las bases para desarrollar análisis exploratorios más profundos y enfrentar de manera adecuada los desafíos "
    "propios de un entorno Big Data."
)

# Bibliografía
doc.add_heading("Bibliografía", level=1)
add_paragraph_with_format(
    doc,
    "• Tukey, J. W. (1977). Exploratory Data Analysis. Addison-Wesley."
)
add_paragraph_with_format(
    doc,
    "• Cleveland, W. S. (1993). Visualizing Data. Hobart Press."
)
add_paragraph_with_format(
    doc,
    "• Wickham, H. (2016). ggplot2: Elegant Graphics for Data Analysis. Springer."
)
add_paragraph_with_format(
    doc,
    "• McKinney, W. (2017). Python for Data Analysis. O'Reilly Media."
)
add_paragraph_with_format(
    doc,
    "• VanderPlas, J. (2016). Python Data Science Handbook. O'Reilly Media."
)

# -----------------------------------------------------------------
# 4. Guardar el documento Word
# -----------------------------------------------------------------
doc.save(output_word_path)
print(f"Documento Word generado exitosamente en: {output_word_path}")

# -----------------------------------------------------------------
# 5. Convertir a PDF usando docx2pdf (opcional)
# -----------------------------------------------------------------
try:
    convert(input_path=output_word_path, output_path=output_pdf_path)
    print(f"Documento PDF generado exitosamente en: {output_pdf_path}")
except Exception as e:
    print("Error al convertir a PDF:", str(e))


Documento Word generado exitosamente en: G:/Mi unidad/FACEN_BIGDATA/Material_Lectura_Unidad2_Extendido.docx


  0%|          | 0/1 [00:00<?, ?it/s]

Documento PDF generado exitosamente en: G:/Mi unidad/FACEN_BIGDATA/Material_Lectura_Unidad2_Extendido.pdf


In [ ]:
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx2pdf import convert

# Para sombrear celdas (en bloques de código)
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# -----------------------------------------------------------------
# Rutas de archivos
# -----------------------------------------------------------------
plantilla_path = "G:/Mi unidad/FACEN_BIGDATA/BigData_plantilla_Guía_Didáctica.docx"
output_word_path = "G:/Mi unidad/FACEN_BIGDATA/Material_Lectura_Unidad3_Python.docx"
output_pdf_path = "G:/Mi unidad/FACEN_BIGDATA/Material_Lectura_Unidad3_Python.pdf"

# -----------------------------------------------------------------
# Funciones auxiliares
# -----------------------------------------------------------------
def add_paragraph_with_format(doc, text, size=11):
    """
    Crea un párrafo con texto justificado e indentación de primera línea.
    """
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    p.paragraph_format.first_line_indent = Inches(0.25)
    run = p.add_run(text)
    run.font.size = Pt(size)
    return p

def add_heading(doc, text, level=1, size=14, bold=True):
    """
    Crea un heading (título) con un cierto nivel y formato.
    """
    heading = doc.add_heading('', level=level)
    run = heading.runs[0]
    run.text = text
    run.font.bold = bold
    run.font.size = Pt(size)
    return heading

def add_list_item(doc, text, size=11):
    """
    Crea un párrafo normal para elementos de lista.
    """
    p = doc.add_paragraph(style='List Bullet')
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    run = p.add_run(text)
    run.font.size = Pt(size)

def add_code_block(doc, code_text, bg_color="EEE8D5", size=9):
    """
    Inserta un recuadro (tabla de 1x1) con fondo color arena (EEE8D5)
    para resaltar bloques de código.
    """
    table = doc.add_table(rows=1, cols=1)
    table.allow_autofit = True
    table.style = 'Table Grid'

    # Sombreado de la celda
    cell = table.cell(0, 0)
    shading_elm = OxmlElement('w:shd')
    shading_elm.set(qn('w:val'), 'clear')
    shading_elm.set(qn('w:color'), 'auto')
    shading_elm.set(qn('w:fill'), bg_color)
    cell._tc.get_or_add_tcPr().append(shading_elm)

    # Añadimos el código en la celda con fuente monoespaciada
    p = cell.add_paragraph()
    run = p.add_run(code_text)
    run.font.name = 'Consolas'
    run.font.size = Pt(size)

    doc.add_paragraph()  # Espacio tras el bloque de código

# -----------------------------------------------------------------
# 1. Crear documento a partir de la plantilla
# -----------------------------------------------------------------
doc = Document(plantilla_path)

# -----------------------------------------------------------------
# 2. Portada adaptada (opcional)
# -----------------------------------------------------------------
add_heading(doc, "Unidad 3: Introducción a Python (Versión Ampliada)", level=1, size=20)
add_paragraph_with_format(
    doc, 
    "Autor: Equipo CursoBigData\nFecha: 2025-03-03\n",
    size=12
)
doc.add_page_break()

# -----------------------------------------------------------------
# 3. Contenido principal
# -----------------------------------------------------------------

# Sección 1
add_heading(doc, "1. ¿Qué es Python?", level=1, size=16)
add_paragraph_with_format(doc,
"""Python es un lenguaje de programación de alto nivel, interpretado y multiparadigma, 
creado originalmente por Guido van Rossum y publicado por primera vez en 1991. Se 
caracteriza por tener una sintaxis limpia y legible, lo que facilita su aprendizaje 
y promueve un código más mantenible.""", 11)

add_heading(doc, "1.1 Historia Breve", level=2, size=14)
add_list_item(doc, "Creador: Guido van Rossum.")
add_list_item(doc, "Año de creación: 1989-1991.")
add_list_item(doc, 'Nombre: inspirado en el grupo de comedia "Monty Python".')
add_list_item(doc, "Fundación Python Software Foundation (PSF): Administra y mantiene el lenguaje.")

add_heading(doc, "1.2 ¿Dónde se descarga y cómo se instala?", level=2, size=14)
add_list_item(doc, "Descarga oficial: https://www.python.org/downloads/")
add_list_item(doc, """Instalación: Disponible para Windows, macOS y Linux.\n - En Windows, basta con ejecutar el instalador .exe.\n - En macOS y Linux, muchas veces está instalado por defecto \n   o se puede instalar con gestores de paquetes.\n""")

add_paragraph_with_format(doc,
"> Nota: Para usar Python en RStudio, se puede utilizar el paquete reticulate.", 10)

# Sección 2
add_heading(doc, "2. Ejecución de Python en RStudio", level=1, size=16)
add_paragraph_with_format(doc,
"""Para ejecutar Python en un documento de R Markdown, se pueden crear 'chunks' de Python \nempleando la sintaxis:\n\n```{python}\n# Código Python\n```\n\nEsto funciona gracias al paquete reticulate, que permite la interoperabilidad entre R y Python.""", 11)

add_heading(doc, "2.1 Ejemplo de un chunk Python en RStudio", level=2, size=14)
code_example_chunk = """print(\"Hola desde Python ejecutándose en RStudio!\")"""
add_code_block(doc, code_example_chunk)

add_paragraph_with_format(doc,
"""A continuación, presentamos varios ejemplos sencillos para familiarizarse con Python.""", 11)

# Sección 3
add_heading(doc, "3. Primeros Pasos en Python", level=1, size=16)
add_paragraph_with_format(doc,
"""En esta sección se presentarán 10 ejemplos para cada tema fundamental: \nOperaciones Básicas, Variables y Tipos de Datos, Estructuras de Control y Funciones.""", 11)

add_heading(doc, "3.1 Operaciones Básicas", level=2, size=14)

op_basicas_1 = """# Suma, Resta, Multiplicación, División\nsuma = 10 + 5\nresta = 10 - 5\nmulti = 10 * 5\ndivision = 10 / 3\n\nprint(\"(1) Suma =\", suma)\nprint(\"(2) Resta =\", resta)\nprint(\"(3) Multiplicación =\", multi)\nprint(\"(4) División =\", division)\n"""
add_heading(doc, "Ejemplo 1: Operadores aritméticos", level=3, size=12)
add_code_block(doc, op_basicas_1)

op_basicas_2 = """potencia = 2 ** 3\nmodulo = 10 % 3\nprint(\"(5) 2^3 =\", potencia)\nprint(\"(6) 10 mod 3 =\", modulo)\n"""
add_heading(doc, "Ejemplo 2: Potencia y Módulo", level=3, size=12)
add_code_block(doc, op_basicas_2)

# Continúa añadiendo más ejemplos, etc. (Para brevedad, no todos pegados aquí)

# Sección 8 (Referencias)
doc.add_page_break()
add_heading(doc, "8. Referencias", level=1, size=16)

add_heading(doc, "8.1 Bibliográficas", level=2, size=14)
add_list_item(doc, "Lutz, M. (2013). Learning Python. O'Reilly Media.")
add_list_item(doc, "Severance, C. (2016). Python for Everybody. CreateSpace.")
add_list_item(doc, "Grus, J. (2015). Data Science from Scratch. O'Reilly Media.")

add_heading(doc, "8.2 Web", level=2, size=14)
add_list_item(doc, "Documentación Oficial de Python: https://docs.python.org/3/")
add_list_item(doc, "W3Schools Tutorial de Python: https://www.w3schools.com/python/")
add_list_item(doc, "Real Python: https://realpython.com/")
add_list_item(doc, "Kaggle Datasets: https://www.kaggle.com/datasets")
add_list_item(doc, "Stack Overflow (python tag): https://stackoverflow.com/questions/tagged/python")

# -----------------------------------------------------------------
# 4. Guardar el documento Word
# -----------------------------------------------------------------
doc.save(output_word_path)
print(f"Documento Word generado exitosamente en: {output_word_path}")

# -----------------------------------------------------------------
# 5. Convertir a PDF usando docx2pdf (opcional)
# -----------------------------------------------------------------
try:
    convert(input_path=output_word_path, output_path=output_pdf_path)
    print(f"Documento PDF generado exitosamente en: {output_pdf_path}")
except Exception as e:
    print("Error al convertir a PDF:", str(e))


IndexError: list index out of range

In [26]:
#pip install reportlab

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 18.2 MB/s eta 0:00:00
